# Voyager 1 in a 1 kHz window
**Biswajit Jana · 365 Earthian Days, Day 7**

Replot an observed Breakthrough Listen/Green Bank Telescope example and measure the slopes of two visible frequency ridges. The data product is identified by its SHA-256.

In [ ]:
import importlib.util, os, sys, subprocess
from pathlib import Path
from urllib.request import urlopen
for name in ("h5py", "hdf5plugin", "matplotlib", "numpy"):
    if importlib.util.find_spec(name) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", name], check=True)
script = Path("research/replay_voyager.py")
if not script.is_file():
    script.parent.mkdir(exist_ok=True)
    base = "https://raw.githubusercontent.com/Biswajit1999/Reading-Voyager-Data-into-Python-and-Plotting/main/"
    with urlopen(base + script.as_posix(), timeout=30) as response:
        script.write_bytes(response.read())
sys.path.insert(0, str(script.parent.resolve()))
import replay_voyager as replay

## Source recording
The example file is about 51 MB. The original archive serves this file over HTTP. Set the VOYAGER_H5 environment variable to use a local copy.

In [ ]:
dataset = Path(os.environ.get("VOYAGER_H5", replay.FILENAME))
replay.acquire(dataset)
print(dataset.name, dataset.stat().st_size, "verified bytes")

## Frequency and time axes
The signed channel spacing in the filterbank header is converted to MHz and reordered for a left-to-right increasing frequency axis. The time axis is relative to the centre of the first integration.

In [ ]:
time, frequency, power, cadence, spacing = replay.read_band(dataset)
print(f"{power.shape[0]} integrations × {power.shape[1]} channels")
print(f"Cadence {cadence:.6f} s, channel spacing {spacing:.6f} Hz")

## Waterfall and slopes
Plot the recorded instrumental power relative to the window median. Within each chosen ridge window, find the peak channel per integration and fit a straight line.

In [ ]:
from IPython.display import Image, display
output = Path("voyager_output")
stats = replay.replay(dataset, output)
display(Image(filename=str(output / "voyager_waterfall.png")))
for row in stats["local_ridges"]:
    print(row["offset_window_hz"], f"{row['apparent_slope_hz_per_s']:.3f} Hz/s")

The slope is a descriptive topocentric measurement of each chosen ridge. The input is the [public Breakthrough Listen example](https://seti.berkeley.edu/opendata).